# Lab 3 — Port a Pure PyTorch Loop to Ray Train (Fundamentals)

**Time:** ~15–20 min  
**Mode:** Complete by hand.

## Learning objectives
1. Wrap a vanilla PyTorch training loop in a `TorchTrainer`.
2. Use `ray.train.torch.prepare_model` to handle device placement / DDP.
3. Report metrics with `ray.train.report` and run with `ScalingConfig(num_workers=2)`.

## What we're training
We use the same `TwoTower` model shape as the course recommender (small
user/item embeddings with a dot-product score). The data here is synthetic so
the lab focuses on the *Ray Train mechanics*, not the model.

(Lab 4 plugs the real Ray Data pipeline in.)

## Setup

In [ ]:
import ray, torch, torch.nn as nn, torch.nn.functional as F

if not ray.is_initialized():
    ray.init()

NUM_USERS, NUM_ITEMS, DIM = 1000, 1000, 64

class TwoTower(nn.Module):
    def __init__(self, num_users, num_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
    def forward(self, u, i):
        u = F.normalize(self.user_emb(u), dim=-1)
        v = F.normalize(self.item_emb(i), dim=-1)
        return u @ v.t()

## Vanilla PyTorch baseline
Run this first — it's the loop you'll port.


In [ ]:
def train_loop_vanilla(epochs=3, batch_size=64):
    model = TwoTower(NUM_USERS, NUM_ITEMS, DIM)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(epochs):
        u = torch.randint(0, NUM_USERS, (batch_size,))
        i = torch.randint(0, NUM_ITEMS, (batch_size,))
        logits = model(u, i)
        loss = F.cross_entropy(logits, torch.arange(batch_size))
        opt.zero_grad(); loss.backward(); opt.step()
        print(f'epoch {epoch}: loss={loss.item():.4f}')
    return model

_ = train_loop_vanilla()

## Exercise 1 — Minimal Ray Train port
Create a `train_loop_ray(config)` that:
1. Builds the model and optimizer just like the vanilla loop.
2. Calls `model = ray.train.torch.prepare_model(model)` after creation.
3. Runs the same per-epoch loop.
4. Calls `ray.train.report({'loss': loss.item()})` each epoch.

Then build a `TorchTrainer` with `ScalingConfig(num_workers=2, use_gpu=False)`
and `.fit()` it.

**Acceptance criteria:**
- The trainer completes without errors.
- `result.metrics_dataframe` shows one row per (worker, epoch).


In [ ]:
# TODO: implement train_loop_ray and run it under TorchTrainer.


## Exercise 2 — A checkpoint
After the last epoch, **save a checkpoint** with the model's `state_dict`
and pass it to `ray.train.report(metrics, checkpoint=...)`.

Use `ray.train.Checkpoint.from_directory(...)` (or the temp-dir helper —
either is fine).

Verify by inspecting `result.checkpoint` after `.fit()`.


In [ ]:
# TODO: add checkpointing to your training loop.


## Wrap-up
- `prepare_model` handles `.to(device)` and DDP wrapping for you.
- `ray.train.report` is how Tune/Trainer learns about your progress and
  collects checkpoints; emit it once per epoch (and again with the final
  checkpoint).
- `ScalingConfig(num_workers=N, use_gpu=True)` is the only thing that needs to
  change to scale across more workers / GPUs.